# Etapa 2 — Preparação das Bases para Modelagem Preditiva

Neste notebook, realizamos a estruturação formal das bases para treinamento supervisionado:
1. **Filtro estrito de pares completos** (`completo_pre` e `completo_pos` simultâneos).
2. **Cálculo da variável-alvo**: `ganho = total_pos - total_pre`.
3. **Estruturação dos Experimentos A e B**:
   - **Experimento A (Acadêmico)**: `total_pre`, `escola`, `serie`, `turma`, `fase`.
   - **Experimento B (Acadêmico + Demográfico/Contextual)**: variáveis do Experimento A mais `sexo`, `faixa_etaria`, `cor_raca`, `zona`, `necessidade_especial`.
4. **Prevenção rigorosa de Data Leakage**: exclusão de `total_pos`, `pct_pos`, `completo_pos`, `responsavel_legal` e uso de `aluno_id` apenas para agrupamento.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from wordgen.modelagem.preparacao import carregar_bases_brutas, preparar_base_instrumento, carregar_e_preparar_dados

# Define diretório raiz do projeto compatível com execução na raiz ou na subpasta
DATA_DIR = Path("data") if Path("data").exists() else Path("../../data")

# Carrega e prepara os 4 datasets experimentais
datasets = carregar_e_preparar_dados(DATA_DIR)
print("Bases preparadas e salvas em data/modelagem/:")
for nome, df in datasets.items():
    print(f"  -> {nome}: {df.shape[0]} linhas x {df.shape[1]} colunas | {df['aluno_id'].nunique()} alunos únicos")

## 2.1. Inspeção da Variável-Alvo (Ganho)

Avaliamos a distribuição do ganho para ambos os instrumentos:

In [ ]:
resumo_ganho = []
for nome, df in datasets.items():
    g = df['ganho']
    resumo_ganho.append({
        'Dataset': nome,
        'N': len(g),
        'Média': round(g.mean(), 3),
        'Mediana': round(g.median(), 3),
        'DP': round(g.std(), 3),
        'Mínimo': round(g.min(), 1),
        'Máximo': round(g.max(), 1),
        '% Ganho > 0': f"{(g > 0).mean():.1%}",
        '% Ganho == 0': f"{(g == 0).mean():.1%}",
        '% Ganho < 0': f"{(g < 0).mean():.1%}",
    })

df_resumo = pd.DataFrame(resumo_ganho)
display(df_resumo)

## 2.2. Verificação de Integridade e Ausências

In [ ]:
for nome, df in datasets.items():
    print(f"=== {nome} ===")
    nulos = df.isna().sum()
    print("Valores ausentes por coluna:")
    print(nulos[nulos > 0] if nulos.sum() > 0 else "Nenhum valor ausente (100% preenchido)")
    print()